# Market, level 2: one step, phase by phase

One step of the real market on the reference scenario, played the way
`LocalEnergyMarket` plays it, with each phase of the
[full cycle](../../docs/full-cycle.md#the-market-step) shown on its own: what it
reads, what it computes, what it hands to the next phase.

[Level 1](market_1_formulas.ipynb) checks each formula on its own numbers. This
level checks the **wiring**: that each phase gets the right inputs from the one
before (this step's trades, this step's power flow, the battery as the last step
left it). Every phase ends with checks of that kind; the last cell lists them.
[Level 3](market_3_days.ipynb) runs the same rules over whole days.

Participants bid with the market's own rule
([`rule_based_bid.net_position`](../../core/market/agent/rule_based_bid.py)):
battery first, then sell the surplus or buy the deficit, at a margin inside the
tariffs. Set `DAY` and `STEP` to look at another step.

In [1]:
import sys
from copy import deepcopy
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "core" / "market").is_dir())
sys.path.insert(0, str(ROOT))

import pandas as pd

from core.config import store
from core.config.scenario import price_at
from core.market.dso import DSO_ID
from core.market.lem import LocalEnergyMarket, OrderIntent, StepRecord

pd.set_option("display.precision", 4)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

TOL = 1e-9
CHECKS = {}


def check(phase: str, name: str, ok: bool, detail: str = "") -> None:
    # Record one check; the detail is kept only when it fails.
    if ok:
        CHECKS[(phase, name)] = ("PASS", "")
    else:
        CHECKS[(phase, name)] = ("FAIL", detail)


def checks(phase: str) -> pd.DataFrame:
    # The checks of one phase, as a table.
    rows = []
    for (p, name), (result, detail) in CHECKS.items():
        if p == phase:
            rows.append({"check": name, "result": result, "detail": detail})
    return pd.DataFrame(rows).set_index("check")

## The scenario

The reference scenario of the market plan (stage 0):
[`validate_market.json`](../../config/scenarios/validate_market.json), five
households on `1-LV-rural1--0-sw`, two with PV and a 10 kWh battery, time-of-use
utility price (MP-0b). This notebook only reads the file; the Studio's Step page
plays the same scenario.

In [2]:
SCENARIO_FILE = ROOT / "config" / "scenarios" / "validate_market.json"
DAY, STEP = "2016-07-01", 9

scenario = store.load(SCENARIO_FILE)   # read only: nothing is saved back
scenario.grid.day = DAY
H = scenario.grid.step_hours
m = scenario.market
d = scenario.dso

print(f"network {scenario.grid.code}, day {DAY}, step {STEP} ({STEP * H:g}:00), "
      f"{H:g} h steps, grid method {scenario.grid.method}")
print(f"mechanism {m.mechanism}, partner preference {m.partner_preference}, "
      f"bid margin {m.bid_margin}, prices {m.min_price}-{m.max_price}")
print(f"feed-in {d.feed_in_tariff} (flat), utility {d.utility_schedule or d.utility_price}, "
      f"fees charged {d.fees_charged}")

# The participants: which SimBench elements each owns, and its battery
rows = []
for p in scenario.participants:
    owns = []
    for element in p.owns:            # e.g. "... sgen 12": keep the last two words
        words = element.split()
        owns.append(words[-2] + " " + words[-1])

    row = {"id": p.id, "bus": p.bus, "owns": ", ".join(owns),
           "battery kWh": None, "power kW": None}
    if p.battery:
        row["battery kWh"] = p.battery.capacity_kwh
        row["power kW"] = p.battery.max_power_kw
    rows.append(row)

pd.DataFrame(rows).set_index("id")

network 1-LV-rural1--0-sw, day 2016-07-01, step 9 (9:00), 1 h steps, grid method powerflow
mechanism average, partner preference True, bid margin 0.3, prices 0.0-1.0
feed-in 0.08 (flat), utility [[0.0, 0.2], [7.0, 0.3], [17.0, 0.4], [22.0, 0.2]], fees charged True


,bus,owns,battery kWh,power kW
id,,,,
P1,LV1.101 Bus 7,"Load 9, SGen 1",10.0,None
P2,LV1.101 Bus 8,"Load 2, SGen 4",NaN,None
P3,LV1.101 Bus 1,Load 8,NaN,None
P4,LV1.101 Bus 5,Load 13,NaN,None
P5,LV1.101 Bus 12,Load 10,10.0,None


## Phase 0: before the day (reset)

[`LocalEnergyMarket.reset`](../../core/market/lem.py) builds each participant
from the SimBench profiles of the elements it took over, puts every battery at
its initial state of charge and every reputation at 0.5, sets the DSO's tariffs
to hour 0 and the reference price $p_\text{ref}$ to the midpoint of the two
tariffs. Then the steps before `STEP` are played, so the step starts from the
state they left: batteries, reputations, $p_\text{ref}$.

In [3]:
lem = LocalEnergyMarket(scenario)
lem.reset()

# The market just after reset
rows = []
for pid, p in lem.participants.items():
    rows.append({
        "id": pid,
        "G today kWh": sum(p.generation_kwh),
        "D today kWh": sum(p.demand_kwh),
        "battery kWh": p.battery.energy_kwh if p.battery else None,
        "reputation": p.reputation,
    })
start = pd.DataFrame(rows).set_index("id")
display(start)

# What reset should give
fit_0 = price_at(d.feed_in_tariff, d.feed_in_schedule, 0)       # hour 0's tariffs
utility_0 = price_at(d.utility_price, d.utility_schedule, 0)

for (pid, p), setup in zip(lem.participants.items(), scenario.participants):
    check("0 reset", f"{pid}: reputation starts at 0.5", p.reputation == 0.5, f"{p.reputation}")
    if p.battery:
        initial = setup.battery.initial_soc * setup.battery.capacity_kwh
        check("0 reset", f"{pid}: battery at its initial charge",
              abs(p.battery.energy_kwh - initial) < TOL, f"{p.battery.energy_kwh} vs {initial}")

check("0 reset", "feed-in tariff of hour 0", lem.dso.feed_in_tariff == fit_0)
check("0 reset", "utility price of hour 0", lem.dso.utility_price == utility_0)
check("0 reset", "p_ref = midpoint of the tariffs",
      abs(lem.clearing_price - (fit_0 + utility_0) / 2) < TOL)

# Play the steps before the one this notebook looks at
for _ in range(STEP):
    lem.step()

rows = []
for pid, p in lem.participants.items():
    rows.append({
        "id": pid,
        "battery kWh": p.battery.energy_kwh if p.battery else None,
        "reputation": p.reputation,
        "profit so far": p.profit,
    })
print(f"After {STEP} steps: p_ref {lem.clearing_price:.4f}, "
      f"tariffs {lem.dso.feed_in_tariff} / {lem.dso.utility_price}")
display(pd.DataFrame(rows).set_index("id"))

checks("0 reset")

,G today kWh,D today kWh,battery kWh,reputation
id,,,,
P1,128.890,23.135,5.0,0.5
P2,62.278,7.133,NaN,0.5
P3,0.000,107.974,NaN,0.5
P4,0.000,107.974,NaN,0.5
P5,0.000,47.832,5.0,0.5


After 9 steps: p_ref 0.1817, tariffs 0.08 / 0.3

,battery kWh,reputation,profit so far
id,,,
P1,9.0,0.5197,1.7512
P2,NaN,0.5610,1.4976
P3,NaN,0.4780,-6.2348
P4,NaN,0.5120,-6.1321
P5,1.0,0.5905,-2.4155


,result,detail
check,,
P1: reputation starts at 0.5,PASS,
P1: battery at its initial charge,PASS,
P2: reputation starts at 0.5,PASS,
P3: reputation starts at 0.5,PASS,
P4: reputation starts at 0.5,PASS,
P5: reputation starts at 0.5,PASS,
P5: battery at its initial charge,PASS,
feed-in tariff of hour 0,PASS,
utility price of hour 0,PASS,


## Phases 1 and 2: inputs and orders

[`start_step`](../../core/market/lem.py) takes the step's inputs ($G_t$, $D_t$,
the tariffs of the hour), asks the bidding rule for each participant's intent
(side, price, quantity, preferred partner), and validates it
([`_validate`](../../core/market/lem.py)): the quantity at most what the
participant can deliver or take (eq. 15, `max_sell_kwh`, `max_buy_kwh`) and
`max_quantity`, the price inside $[p_{\min}, p_{\max}]$. What is left becomes an
order. $B_\text{grid}$, offered minus asked, is the market's balance before
stage 3.

In [4]:
before = deepcopy(lem)       # the market just before the step, for the example below
pending = lem.start_step()   # phases 1 and 2: the inputs, the intents, the orders
inp = pending.inputs

orders = {}                  # participant -> its order (at most one each)
for o in pending.orders:
    orders[o.agent_id] = o

# Each participant: its inputs, its eq. 15 bounds, what it wanted and what it ordered
rows = []
for pid, p in lem.participants.items():
    intent = pending.intents.get(pid)   # what the bidder wanted
    order = orders.get(pid)             # what was accepted, after validation
    row = {"id": pid, "G": inp.generation_kwh[pid], "D": inp.demand_kwh[pid],
           "max sell": p.max_sell_kwh(STEP), "max buy": p.max_buy_kwh(STEP),
           "side": "-", "intent kWh": None, "order kWh": None, "price": None, "partner": None}
    if intent:
        row["intent kWh"] = intent.quantity_kwh
    if order:
        row["side"] = "buy" if order.is_buy else "sell"
        row["order kWh"] = order.quantity
        row["price"] = order.price
        row["partner"] = order.partner_id
    rows.append(row)

# B_grid by hand: what is offered minus what is asked
offered, asked = 0.0, 0.0
for o in pending.orders:
    if o.is_buy:
        asked += o.quantity
    else:
        offered += o.quantity
b_grid = offered - asked

print(f"tariffs: FiT {inp.feed_in_tariff}, utility {inp.utility_price};  "
      f"B_grid {pending.grid_balance_kwh:+.3f} kWh")
display(pd.DataFrame(rows).set_index("id"))

# The checks
hour = STEP * H
fit = price_at(d.feed_in_tariff, d.feed_in_schedule, hour)
utility = price_at(d.utility_price, d.utility_schedule, hour)
check("1-2 orders", "feed-in tariff of this hour", inp.feed_in_tariff == fit)
check("1-2 orders", "utility price of this hour (time of use)", inp.utility_price == utility)

for pid, p in lem.participants.items():
    same_g = inp.generation_kwh[pid] == p.generation_kwh[STEP]
    same_d = inp.demand_kwh[pid] == p.demand_kwh[STEP]
    check("1-2 orders", f"{pid}: G and D are its profiles at this step", same_g and same_d)

for pid, o in orders.items():
    p = lem.participants[pid]
    if o.is_buy:
        limit = p.max_buy_kwh(STEP)
    else:
        limit = p.max_sell_kwh(STEP)
    check("1-2 orders", f"{pid}: order within eq. 15", o.quantity <= limit + TOL, f"{o.quantity} > {limit}")
    check("1-2 orders", f"{pid}: price within [p_min, p_max]",
          m.min_price <= o.price <= m.max_price, f"{o.price}")

check("1-2 orders", "B_grid = offered - asked", abs(pending.grid_balance_kwh - b_grid) < TOL)
checks("1-2 orders")

tariffs: FiT 0.08, utility 0.3;  B_grid +12.847 kWh

,G,D,max sell,max buy,side,intent kWh,order kWh,price,partner
id,,,,,,,,,
P1,19.899,1.405,26.094,0.0000,sell,18.494,18.494,0.1444,P5
P2,9.888,0.424,9.464,0.0000,sell,9.464,9.464,0.1404,P5
P3,0.000,6.555,0.000,6.5550,buy,6.555,6.555,0.2181,P2
P4,0.000,6.555,0.000,6.5550,buy,6.555,6.555,0.2501,P1
P5,0.000,2.001,0.000,10.4221,buy,2.001,2.001,0.2123,P1


,result,detail
check,,
feed-in tariff of this hour,PASS,
utility price of this hour (time of use),PASS,
P1: G and D are its profiles at this step,PASS,
P2: G and D are its profiles at this step,PASS,
P3: G and D are its profiles at this step,PASS,
P4: G and D are its profiles at this step,PASS,
P5: G and D are its profiles at this step,PASS,
P1: order within eq. 15,PASS,
"P1: price within [p_min, p_max]",PASS,


### What validation does with an order it cannot accept

On a copy of the market at the same moment, three intents the rule would never
send: P1 offers 999 kWh at 2.0 (more than it has, above $p_{\max}$), P3 (a load
only, nothing to sell) offers 1 kWh, P4 bids at $-0.5$ (below $p_{\min}$). This is
market plan stage 2 (MP-2: invalid actions, M-02).

In [5]:
demo = deepcopy(before)   # a copy: the real market above is not touched

# Three intents the rule would never send: OrderIntent(participant, is_buy, price, kWh)
odd = {
    "P1": OrderIntent("P1", False, 2.0, 999.0),   # sells 999 kWh at 2.0
    "P3": OrderIntent("P3", False, 0.10, 1.0),    # sells 1 kWh it does not have
    "P4": OrderIntent("P4", True, -0.5, 1.0),     # buys at a negative price
}
demo_pending = demo.start_step(odd)
print("validation notes:", *demo_pending.notes, sep="\n  ")

demo_orders = {}
for o in demo_pending.orders:
    demo_orders[o.agent_id] = o

rows = []
for pid, o in demo_orders.items():
    rows.append({"id": pid, "kWh": o.quantity, "price": o.price})
display(pd.DataFrame(rows).set_index("id"))

p1_limit = demo.participants["P1"].max_sell_kwh(STEP)
check("1-2 orders", "too large: cut to eq. 15", abs(demo_orders["P1"].quantity - p1_limit) < TOL)
check("1-2 orders", "price above p_max: set to p_max", demo_orders["P1"].price == m.max_price)
check("1-2 orders", "nothing to deliver: no order", "P3" not in demo_orders)
check("1-2 orders", "price below p_min: set to p_min", demo_orders["P4"].price == m.min_price)

validation notes:

P1: 999.00 kWh limited to 26.09

P1: price 2.000 limited to 1.000

P3: 1.00 kWh limited to 0.00

P4: price -0.500 limited to 0.000

,kWh,price
id,,
P1,26.094,1.0
P4,1.000,0.0


## Phase 3: clearing, stage by stage

Each [`advance`](../../core/market/lem.py) runs the next stage on the orders the
previous one left open ([`clearing.py`](../../core/market/clearing.py)): mutual
preferences, the double auction, then the DSO. Every trade leaves its stage with
its distance $d_{ij}$ along the lines and its loss $l = d\,q\,\kappa$ (eq. 18);
its fee comes after the power flow (phase 6).

In [6]:
def trades_table(trades):
    rows = []
    for t in trades:
        rows.append({"stage": t.stage, "seller": t.seller_id, "buyer": t.buyer_id, "kWh": t.quantity,
                     "price": t.price, "km": t.distance, "loss kWh": t.transmission_loss})
    return pd.DataFrame(rows, columns=["stage", "seller", "buyer", "kWh", "price", "km", "loss kWh"])


# Run the three stages one at a time
stages = []
for _ in range(3):
    s = lem.advance()
    stages.append(s)

    still_open = []
    for o in s.remaining:
        still_open.append(f"{o.agent_id} {o.quantity:.3f}")
    print(f"--- stage {s.stage} ({s.name}): {len(s.trades)} trades, {s.volume_kwh:.3f} kWh; "
          f"open after it: {', '.join(still_open) or 'none'}")
    display(trades_table(s.trades))

stage_1, stage_2, stage_3 = stages
trades = stage_1.trades + stage_2.trades + stage_3.trades
peer = stage_1.trades + stage_2.trades           # the trades between participants

# Every order traded in full (stage 3 takes whatever is left)
for pid, o in orders.items():
    done = 0.0
    for t in trades:
        if o.is_buy and t.buyer_id == pid:
            done += t.quantity
        if not o.is_buy and t.seller_id == pid:
            done += t.quantity
    check("3 clearing", f"{pid}: traded = its order", abs(done - o.quantity) < 1e-6, f"{done} of {o.quantity}")

# Stage 1: only pairs that named each other
for t in stage_1.trades:
    buyer_named_seller = orders[t.buyer_id].partner_id == t.seller_id
    seller_named_buyer = orders[t.seller_id].partner_id == t.buyer_id
    check("3 clearing", f"stage 1, {t.seller_id}->{t.buyer_id}: they named each other",
          buyer_named_seller and seller_named_buyer)

# Every peer trade: the bids cross, the price is their average, the loss is eq. 18
for t in peer:
    bid = orders[t.buyer_id].price
    ask = orders[t.seller_id].price
    path_km = lem._path(t.buyer_id, t.seller_id).distance_km
    name = f"stage {t.stage}, {t.seller_id}->{t.buyer_id}"
    check("3 clearing", f"{name}: bid >= ask", bid >= ask, f"{bid} < {ask}")
    check("3 clearing", f"{name}: price = (bid + ask) / 2", abs(t.price - (bid + ask) / 2) < TOL)
    check("3 clearing", f"{name}: distance of its path", abs(t.distance - path_km) < TOL)
    check("3 clearing", f"{name}: loss = d q kappa",
          abs(t.transmission_loss - t.distance * t.quantity * d.loss_factor) < TOL)

# Stage 3: the DSO takes up the imbalance, and nothing stays open
dso_bought, dso_sold = 0.0, 0.0
for t in stage_3.trades:
    if t.buyer_id == DSO_ID:
        dso_bought += t.quantity
    if t.seller_id == DSO_ID:
        dso_sold += t.quantity
check("3 clearing", "the DSO takes up B_grid (bought - sold)",
      abs((dso_bought - dso_sold) - pending.grid_balance_kwh) < 1e-6)
check("3 clearing", "nothing open after stage 3", len(stage_3.remaining) == 0)

checks("3 clearing")

--- stage 1 (Preferences): 1 trades, 2.001 kWh; open after it: P1 16.493, P2 9.464, P3 6.555, P4 6.555

,stage,seller,buyer,kWh,price,km,loss kWh
0,1,P1,P5,2.001,0.1783,0.0021,0.0001


--- stage 2 (Double auction): 3 trades, 13.110 kWh; open after it: P1 12.847

,stage,seller,buyer,kWh,price,km,loss kWh
0,2,P2,P4,6.555,0.1953,0.2505,0.0493
1,2,P2,P3,2.909,0.1793,0.1376,0.0120
2,2,P1,P3,3.646,0.1812,0.1823,0.0199


--- stage 3 (DSO): 1 trades, 12.847 kWh; open after it: none

,stage,seller,buyer,kWh,price,km,loss kWh
0,3,P1,DSO,12.847,0.08,0.0498,0.0192


,result,detail
check,,
P1: traded = its order,PASS,
P2: traded = its order,PASS,
P3: traded = its order,PASS,
P4: traded = its order,PASS,
P5: traded = its order,PASS,
"stage 1, P1->P5: they named each other",PASS,
"stage 1, P1->P5: bid >= ask",PASS,
"stage 1, P1->P5: price = (bid + ask) / 2",PASS,
"stage 1, P1->P5: distance of its path",PASS,


## Phase 4: settlement

The next `advance` settles the step, runs the power flow, charges the fees and
records it (phases 4 to 8). Each participant turns its trades into what it
physically did ([`Participant.settle`](../../core/market/agent/der.py)): its
balance $B = (G - D) + (\text{bought} - \text{sold})$ goes into or out of the
battery, the rest is exchanged with the upstream grid untraded; its profit is
what it sold minus what it bought, then minus its fees (phase 6).

In [7]:
# Remember the state before settlement
energy_before, reputation_before, profit_before = {}, {}, {}
for pid, p in lem.participants.items():
    energy_before[pid] = p.battery.energy_kwh if p.battery else None
    reputation_before[pid] = p.reputation
    profit_before[pid] = p.profit

record = lem.advance()          # phases 4 to 8 in one go
assert isinstance(record, StepRecord)
outcomes = record.participants  # participant -> its outcome this step

rows = []
for o in outcomes.values():
    rows.append(vars(o))
display(pd.DataFrame(rows).set_index("participant_id"))

for pid, o in outcomes.items():
    battery = lem.participants[pid].battery

    # Its trades, by hand
    bought, sold, value = 0.0, 0.0, 0.0
    for t in record.trades:
        if t.buyer_id == pid:
            bought += t.quantity
            value -= t.price * t.quantity
        if t.seller_id == pid:
            sold += t.quantity
            value += t.price * t.quantity

    check("4 settlement", f"{pid}: bought = its trades", abs(o.bought_kwh - bought) < TOL)
    check("4 settlement", f"{pid}: sold = its trades", abs(o.sold_kwh - sold) < TOL)

    if battery:
        expected = (energy_before[pid]
                    + o.charge_kwh * battery.charge_efficiency
                    - o.discharge_kwh / battery.discharge_efficiency)
        check("4 settlement", f"{pid}: battery = before + charge·η - discharge/η",
              abs(o.soc_kwh - expected) < 1e-9, f"{o.soc_kwh} vs {expected}")

    check("4 settlement", f"{pid}: profit = trade value - fees (eq. 14)",
          abs(o.profit - (value - o.fees)) < TOL)
    check("4 settlement", f"{pid}: running profit grew by the step's",
          abs(lem.participants[pid].profit - (profit_before[pid] + o.profit)) < TOL)

checks("4 settlement")

,generation_kwh,demand_kwh,bought_kwh,sold_kwh,charge_kwh,discharge_kwh,external_kwh,soc_kwh,balance_kwh,fees,profit
participant_id,,,,,,,,,,,
P1,19.899,1.405,0.000,18.494,0.0,0.0,0.0,9.0,0.0,-0.0066,2.0520
P2,9.888,0.424,0.000,9.464,0.0,0.0,0.0,NaN,0.0,0.0353,1.7662
P3,0.000,6.555,6.555,0.000,0.0,0.0,0.0,NaN,0.0,0.0173,-1.1996
P4,0.000,6.555,6.555,0.000,0.0,0.0,0.0,NaN,0.0,0.0289,-1.3089
P5,0.000,2.001,2.001,0.000,0.0,0.0,0.0,1.0,0.0,-0.0267,-0.3301


,result,detail
check,,
P1: bought = its trades,PASS,
P1: sold = its trades,PASS,
P1: battery = before + charge·η - discharge/η,PASS,
P1: profit = trade value - fees (eq. 14),PASS,
P1: running profit grew by the step's,PASS,
P2: bought = its trades,PASS,
P2: sold = its trades,PASS,
P2: profit = trade value - fees (eq. 14),PASS,
P2: running profit grew by the step's,PASS,


## Phase 5: the grid

Each participant's power at its bus from what it physically did
([`injection`](../../core/market/agent/der.py)): $(G + \text{discharge} - D -
\text{charge})/\Delta t$, never its trades. The network's other loads and
generators follow their SimBench profiles; one AC power flow
([`SimBenchGrid.assess`](../../core/grid/powerflow.py)).

In [8]:
rows = []
for pid, o in outcomes.items():
    inj = lem.participants[pid].injection(o, H)
    # By hand: what it physically produced minus what it consumed, as power
    by_rule = (o.generation_kwh + o.discharge_kwh - o.demand_kwh - o.charge_kwh) / H
    rows.append({"id": pid, "production kW": inj.production_kw, "consumption kW": inj.consumption_kw,
                 "net kW": inj.net_injection_kw, "(G + dis - D - ch)/h": by_rule})
    check("5 grid", f"{pid}: injection = (G + dis - D - ch)/h", abs(inj.net_injection_kw - by_rule) < TOL)
display(pd.DataFrame(rows).set_index("id"))

g = record.grid
print(f"power flow converged {g.converged}; voltage {g.min_voltage_pu:.3f}-{g.max_voltage_pu:.3f} pu")
print(f"max line loading {g.max_line_loading_percent} %; losses {g.losses_kwh} kWh; "
      f"from the upstream grid {g.external_grid_kwh:+} kWh; violations {len(g.violations)}")
check("5 grid", "the power flow converges", g.converged)
checks("5 grid")

,production kW,consumption kW,net kW,(G + dis - D - ch)/h
id,,,,
P1,19.899,1.405,18.494,18.494
P2,9.888,0.424,9.464,9.464
P3,0.000,6.555,-6.555,-6.555
P4,0.000,6.555,-6.555,-6.555
P5,0.000,2.001,-2.001,-2.001


power flow converged True; voltage 1.025-1.029 pu

max line loading 15.55 %; losses 0.678 kWh; from the upstream grid -40.088 kWh; violations 0

,result,detail
check,,
P1: injection = (G + dis - D - ch)/h,PASS,
P2: injection = (G + dis - D - ch)/h,PASS,
P3: injection = (G + dis - D - ch)/h,PASS,
P4: injection = (G + dis - D - ch)/h,PASS,
P5: injection = (G + dis - D - ch)/h,PASS,
the power flow converges,PASS,


## Phase 6: the fees

The DSO looks at the line loading of **this step's** power flow (with
`grid.method = "graph"`, the graph model's, last section), and gives every trade
its fee (eq. 8; the parts are checked one by one in level 1, sections 10-11).
$C_t$ is the highest loading on the trade's path: a peer trade's path between its
two buses, a DSO trade's path to the transformer.

In [9]:
def highest_loading(loading_percent, lines):
    # C_t: the highest loading on the trade's path, 0 to 1 (0 when it uses no line)
    highest = 0.0
    for line in lines:
        highest = max(highest, loading_percent.get(line, 0.0))
    return min(1.0, highest / 100)


rows = []
for t in record.trades:
    name = f"{t.seller_id}->{t.buyer_id}"
    path = lem.dso.trade_path(t, lem._path, lem._upstream)
    c_flow = highest_loading(g.line_loading_percent, path.lines)   # from this step's power flow
    c_dso = lem.dso.congestion(path.lines)                         # what the DSO used
    p = t.fee_parts
    peer_trade = t.stage < 3

    rows.append({"stage": t.stage, "seller": t.seller_id, "buyer": t.buyer_id, "C power flow": c_flow,
                 "C the DSO used": c_dso, "same zone": path.same_zone, "cong": p.congestion,
                 "thermal": p.thermal, "volt": p.voltage, "trans": p.transmission, "zone": p.zone,
                 "imb": p.imbalance, "total": t.fees, "buyer pays": t.buyer_fee, "seller pays": t.seller_fee})

    check("6 fees", f"{name}: fees read this step's power flow", abs(c_flow - c_dso) < TOL)
    check("6 fees", f"{name}: congestion only above C_th",
          (p.congestion > 0) == (c_dso > d.congestion_threshold))
    check("6 fees", f"{name}: thermal only above T_th",
          (p.thermal > 0) == (c_dso > d.thermal_threshold))
    check("6 fees", f"{name}: zone bonus only for a peer trade in one zone",
          (p.zone < 0) == (peer_trade and path.same_zone))
    check("6 fees", f"{name}: fee = sum of its parts", abs(t.fees - p.total) < TOL)
    due = t.fees if d.fees_charged else 0.0
    check("6 fees", f"{name}: fee paid once (buyer + seller share)",
          abs(t.buyer_fee + t.seller_fee - due) < TOL)
display(pd.DataFrame(rows))

# Each participant pays its share of every trade it is in
for pid, o in outcomes.items():
    share = 0.0
    for t in record.trades:
        if t.buyer_id == pid:
            share += t.buyer_fee
        if t.seller_id == pid:
            share += t.seller_fee
    check("6 fees", f"{pid}: pays its share of every trade it is in", abs(o.fees - share) < TOL)

checks("6 fees")

,stage,seller,buyer,C power flow,C the DSO used,same zone,cong,thermal,volt,trans,zone,imb,total,buyer pays,seller pays
0,1,P1,P5,0.0516,0.0516,True,0.0,0.0,0.0001,2.2983e-05,-0.0535,0.0,-0.0534,-0.0267,-0.0267
1,2,P2,P4,0.1555,0.1555,False,0.0,0.0,0.0481,9.6186e-03,0.0000,0.0,0.0577,0.0289,0.0289
2,2,P2,P3,0.1555,0.1555,False,0.0,0.0,0.0108,2.1533e-03,0.0000,0.0,0.0129,0.0065,0.0065
3,2,P1,P3,0.0448,0.0448,False,0.0,0.0,0.0181,3.6141e-03,0.0000,0.0,0.0217,0.0108,0.0108
4,3,P1,DSO,0.0448,0.0448,False,0.0,0.0,0.0077,1.5359e-03,0.0000,0.0,0.0092,0.0000,0.0092


,result,detail
check,,
P1->P5: fees read this step's power flow,PASS,
P1->P5: congestion only above C_th,PASS,
P1->P5: thermal only above T_th,PASS,
P1->P5: zone bonus only for a peer trade in one zone,PASS,
P1->P5: fee = sum of its parts,PASS,
P1->P5: fee paid once (buyer + seller share),PASS,
P2->P4: fees read this step's power flow,PASS,
P2->P4: congestion only above C_th,PASS,
P2->P4: thermal only above T_th,PASS,


## Phase 7: the record

The step's clearing price, the next step's $p_\text{ref}$, is the volume-weighted
price of the peer trades (the previous one when there were none). The DSO
settles: eq. 7, its balance sold minus bought, its revenue and the fees it
collected. Every unit of money leaves one party and reaches another.

In [10]:
# By hand: the volume-weighted price of the peer trades
peer_value, peer_kwh = 0.0, 0.0
for t in peer:
    peer_value += t.price * t.quantity
    peer_kwh += t.quantity
vwap = peer_value / peer_kwh if peer else None

dso = record.dso
print(f"clearing price {record.clearing_price:.4f} (volume-weighted peer price {vwap})")
print(f"DSO: bought {dso.bought_kwh:.3f}, sold {dso.sold_kwh:.3f}, balance {dso.balance_kwh:+.3f} kWh; "
      f"energy revenue {dso.energy_revenue:+.4f}, fees {dso.fees:.4f}")

if peer:
    check("7 record", "clearing price = volume-weighted peer price", abs(record.clearing_price - vwap) < TOL)
check("7 record", "DSO balance = sold - bought (eq. 7)",
      abs(dso.balance_kwh - (dso.sold_kwh - dso.bought_kwh)) < TOL)
check("7 record", "DSO balance = -B_grid", abs(dso.balance_kwh + record.grid_balance_kwh) < 1e-6)

# Money only moves between the participants and the DSO, so it sums to zero
paid = dso.profit
for o in outcomes.values():
    paid += o.profit
check("7 record", "money adds up: participants + DSO = 0", abs(paid) < 1e-9, f"{paid}")

checks("7 record")

clearing price 0.1866 (volume-weighted peer price 0.18656109613650249)

DSO: bought 12.847, sold 0.000, balance -12.847 kWh; energy revenue -1.0278, fees 0.0481

,result,detail
check,,
clearing price = volume-weighted peer price,PASS,
DSO balance = sold - bought (eq. 7),PASS,
DSO balance = -B_grid,PASS,
money adds up: participants + DSO = 0,PASS,


## Phase 8: reputation and the next step

Each participant's reputation from this step (eq. 12, level 1 section 14), blended
into the old one with the decay ($0.8$ per hour, as $0.8^{\Delta t}$ per step,
M-43). Then the clock moves on: the next hour's tariffs, $p_\text{ref}$ = this
step's clearing price.

In [11]:
decay = m.reputation_decay ** H   # the decay per step (per hour, raised to the step length)

rows = []
for pid, u in record.reputation.items():
    rows.append({"id": pid, "before": reputation_before[pid], "reliability": u.reliability,
                 "fairness": u.fairness, "grid": u.grid, "score": u.score, "after": u.reputation})
    expected = decay * reputation_before[pid] + (1 - decay) * u.score
    check("8 advance", f"{pid}: reputation = decay·before + (1 - decay)·score",
          abs(u.reputation - expected) < TOL)
display(pd.DataFrame(rows).set_index("id"))

next_hour = (STEP + 1) * H
check("8 advance", "the clock moved one step", lem.current_step == STEP + 1)
check("8 advance", "next hour's feed-in tariff",
      lem.dso.feed_in_tariff == price_at(d.feed_in_tariff, d.feed_in_schedule, next_hour))
check("8 advance", "next hour's utility price",
      lem.dso.utility_price == price_at(d.utility_price, d.utility_schedule, next_hour))
check("8 advance", "next p_ref = this step's clearing price", lem.clearing_price == record.clearing_price)

checks("8 advance")

,before,reliability,fairness,grid,score,after
id,,,,,,
P1,0.5197,0.3053,0.9637,0.4267,0.4977,0.5153
P2,0.5610,1.0000,0.9571,0.3158,0.6493,0.5786
P3,0.4780,1.0000,0.9662,0.6276,0.8070,0.5438
P4,0.5120,1.0000,0.9533,0.7551,0.8682,0.5832
P5,0.5905,1.0000,0.9559,0.5779,0.7801,0.6285


,result,detail
check,,
P1: reputation = decay·before + (1 - decay)·score,PASS,
P2: reputation = decay·before + (1 - decay)·score,PASS,
P3: reputation = decay·before + (1 - decay)·score,PASS,
P4: reputation = decay·before + (1 - decay)·score,PASS,
P5: reputation = decay·before + (1 - decay)·score,PASS,
the clock moved one step,PASS,
next hour's feed-in tariff,PASS,
next hour's utility price,PASS,
next p_ref = this step's clearing price,PASS,


## The same step with the graph model

With `grid.method = "graph"` the only thing that changes is the line loading the
fees read: the graph model's (each trade's power along its path, over a capacity
per line; level 1 section 9) instead of the power flow's. The bidding rule
ignores fees, so the orders, the trades and the physics (the power flow, still
run and recorded) must come out the same; only the fees differ.

In [12]:
# The same scenario and step, with the graph model's line loading for the fees
graph = deepcopy(scenario)
graph.grid.method = "graph"
lem_g = LocalEnergyMarket(graph)
lem_g.reset()
for _ in range(STEP):
    lem_g.step()
record_g = lem_g.step()


def trade_key(t):
    return (t.stage, t.seller_id, t.buyer_id, round(t.quantity, 9), round(t.price, 9))


same_trades = len(record.trades) == len(record_g.trades)
for t, tg in zip(record.trades, record_g.trades):
    if trade_key(t) != trade_key(tg):
        same_trades = False

check("graph", "same trades", same_trades)
check("graph", "same power flow", record.grid.line_loading_percent == record_g.grid.line_loading_percent)
check("graph", "the fees read the graph's loading", record_g.signal is not None)

rows = []
for t, tg in zip(record.trades, record_g.trades):
    path = lem.dso.trade_path(t, lem._path, lem._upstream)
    rows.append({"stage": t.stage, "seller": t.seller_id, "buyer": t.buyer_id, "kWh": t.quantity,
                 "C power flow": highest_loading(record.grid.line_loading_percent, path.lines),
                 "C graph": highest_loading(record_g.signal.line_loading_percent, path.lines),
                 "fee power flow": t.fees, "fee graph": tg.fees})
display(pd.DataFrame(rows))

checks("graph")

,stage,seller,buyer,kWh,C power flow,C graph,fee power flow,fee graph
0,1,P1,P5,2.001,0.0516,0.0103,-0.0534,-0.0534
1,2,P2,P4,6.555,0.1555,0.6425,0.0577,0.4417
2,2,P2,P3,2.909,0.1555,0.4860,0.0129,0.1694
3,2,P1,P3,3.646,0.0448,0.6425,0.0217,0.2199
4,3,P1,DSO,12.847,0.0448,0.6425,0.0092,0.3175


,result,detail
check,,
same trades,PASS,
same power flow,PASS,
the fees read the graph's loading,PASS,


## Summary

Every check of this step, by phase. A FAIL names what did not hold.

In [13]:
rows = []
for (phase, name), (result, detail) in CHECKS.items():
    rows.append({"phase": phase, "check": name, "result": result, "detail": detail})
summary = pd.DataFrame(rows)

passed = (summary.result == "PASS").sum()
print(f"{passed} of {len(summary)} checks pass")
summary.groupby("phase", sort=False).result.value_counts().unstack(fill_value=0)

135 of 135 checks pass

result,PASS
phase,
0 reset,10
1-2 orders,22
3 clearing,24
4 settlement,22
5 grid,6
6 fees,35
7 record,4
8 advance,9
graph,3


In [14]:
summary[summary.result == "FAIL"]

,phase,check,result,detail
